In [ ]:
!pip install --upgrade pip
!pip install datasets
!pip install transformers
!pip install nltk

In [ ]:
# 2.1 Controlling the Lengths

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

import random
from datasets import load_dataset
import nltk
from nltk.tokenize import word_tokenize

from torch.utils.data import Dataset, DataLoader

from transformers import Trainer, TrainingArguments
from sklearn.model_selection import train_test_split

import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.environ["WANDB_DISABLED"] = "true"

nltk.download('punkt_tab')
#1
def load_model_and_tokenizer():
    """Initialize GPT-2 model and tokenizer with special tokens for control."""

    #Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained('gpt2')
    model = AutoModelForCausalLM.from_pretrained('gpt2')

    #Define our special control tokens
    special_tokens = {'additional_special_tokens': ['<len>', '<text>']}

    #Add them to the tokenizer
    tokenizer.add_special_tokens(special_tokens)

    #Resize the model embeddings to include the new tokens
    model.resize_token_embeddings(len(tokenizer))

    #Set padding token cuz GPT-2 doesn't have one by default)
    tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer

#Check
model, tokenizer = load_model_and_tokenizer()
print("Model and tokenizer loaded successfully.")

#2

def process_dataset(percentage=0.001):
    """Loads a small portion of OpenWebText dataset and processes it for training."""

    #Load dataset
    dataset = load_dataset("Bingsu/openwebtext_20p", split="train")

    #Select a random subset of data
    num_samples = len(dataset)
    indices = random.sample(range(num_samples), int(percentage * num_samples))
    subset = dataset.select(indices)

    processed_texts = []

    for text in subset['text']:
        words = word_tokenize(text)#Tokenize words to count them
        word_count = len(words)

        #Format with our special tokens
        processed_text = f"<len> {word_count} <text> " + text
        processed_texts.append(processed_text)

    return processed_texts

#Run and check dataset
train_texts = process_dataset()
print(f"Processed {len(train_texts)} training samples.")

#3
class LengthControlDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.texts = texts
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        tokenized = self.tokenizer(
            text,
            truncation=True,
            padding="max_length", #Padding Strategy
            max_length=self.max_length,
            return_tensors="pt"
        )

        #Remove batch dimension that the tokenizer adds by default
        for key in tokenized:
            tokenized[key] = tokenized[key].squeeze(0)

        #Setup labels for clm
        tokenized['labels'] = tokenized['input_ids'].clone()

        #Mask padding tokens in labels
        tokenized['labels'] = tokenized['labels'].masked_fill(
            tokenized['labels'] == self.tokenizer.pad_token_id, -100
        )

        return tokenized

#Run and check tokenization
dataset = LengthControlDataset(train_texts, tokenizer)
print("Sample tokenized output:", dataset[0])

#4

#Split dataset (80/20)
train_texts, val_texts = train_test_split(train_texts, test_size=0.2, random_state=42)

#Convert the datasets to PyTorch datasets
train_dataset = LengthControlDataset(train_texts, tokenizer)
val_dataset = LengthControlDataset(val_texts, tokenizer)

def fine_tune_model(model, tokenizer, train_texts, batch_size=8, epochs=3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"Model is on device: {model.device}")

    dataset = LengthControlDataset(train_texts, tokenizer)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=False)

    if hasattr(model, 'config'):
        model.config.use_cache = False

    training_args = TrainingArguments(
        output_dir="./gpt2_length_control",
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        save_steps=1000,
        save_total_limit=1,
        logging_dir="./logs",
        logging_steps=50,
        learning_rate=3e-5,
        evaluation_strategy="epoch",
        fp16=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        eval_dataset=val_dataset
    )

    trainer.train()


# Execute fine-tuning
fine_tune_model(model, tokenizer, train_texts)

#5. Evaluation
def evaluate_model(model, tokenizer):
    """Generates text with different length prompts and evaluates control accuracy."""

    target_lengths = [5, 10, 15, 20, 25, 30]  #Desired lengths
    errors = []

    for length in target_lengths:
        prompt = f"<len> {length} <text>"
        input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
        output = model.generate(input_ids.to(device), attention_mask=input_ids.ne(tokenizer.pad_token_id), max_length=length+5, min_length= length-5, do_sample=True, temperature=0.7)
        generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

        generated_length = len(word_tokenize(generated_text))
        errors.append(abs(length - generated_length))

        print(f"Target: {length}, Generated: {generated_length}, Text: {generated_text}")

    avg_error = sum(errors) / len(errors)
    print(f"Average length control error: {avg_error:.2f}")

# Run evaluation
evaluate_model(model, tokenizer)

In [ ]:
# 2.2 Custom Control Property: Sentiment Tokens

import os
import random
import torch
import nltk
from nltk.tokenize import word_tokenize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
nltk.download('punkt_tab')
os.environ["WANDB_DISABLED"] = "true"

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments
)

#1 Load Model & Tokenizer
def load_model_and_tokenizer():
    """Initialize GPT-2 model and tokenizer with special sentiment tokens."""

    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    model = AutoModelForCausalLM.from_pretrained("gpt2")

    #Special token <sentiment> -> 'positive' or 'negative'
    special_tokens = {"additional_special_tokens": ["<sentiment>", "<text>"]}
    tokenizer.add_special_tokens(special_tokens)

    #Resize the embeddings
    model.resize_token_embeddings(len(tokenizer))

    #Set padding token cuz GPT-2 doesn't have one by default
    tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer

model, tokenizer = load_model_and_tokenizer()
model.to(device)
model.gradient_checkpointing_enable()

print("Model and tokenizer loaded successfully.")


#2 Dataset Prep (Yelp Polarity)
def create_sentiment_dataset(sample_size=10000, seed=42):
    """
    Loads a subset of the Yelp Polarity dataset, converts label 0->negative, 1->positive,
    and prepends each example with <sentiment> negative/positive <text>.
    Returns a list of processed strings.
    """
    random.seed(seed)

    #Load Yelp Polarity dataset from Hugging Face
      #Has labels (0=negative, 1=positive) and "text"
    yelp = load_dataset("yelp_polarity", split="train")

    #Shuffle and Take a smaller subset
    yelp = yelp.shuffle(seed=seed).select(range(sample_size))

    processed_texts = []
    for entry in yelp:
        label = entry["label"]  # 0 or 1
        text = entry["text"]

        sentiment_str = "positive" if label == 1 else "negative"

        # Prepend with the sentiment control token
        # Example: "<sentiment> positive <text> Great restaurant..."
        processed = f"<sentiment> {sentiment_str} <text> {text}"
        processed_texts.append(processed)

    return processed_texts

all_texts = create_sentiment_dataset(sample_size=10000)
print(f"Prepared {len(all_texts)} labeled samples from Yelp Polarity.")


#3 Dataset Class for Tokenization
class SentimentControlDataset(Dataset):
    """Tokenizes text with sentiment control tokens for CLM"""
    def __init__(self, texts, tokenizer, max_length=512):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        tokenized = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        # Create labels for CLM
        tokenized["labels"] = tokenized["input_ids"].clone()

        # Replace pad token with -100
        tokenized["labels"][tokenized["labels"] == self.tokenizer.pad_token_id] = -100

        return {k: v.squeeze(0) for k, v in tokenized.items()}

#4 Train/Val Split and Trainer

train_texts, val_texts = train_test_split(all_texts, test_size=0.2, random_state=42)
train_dataset = SentimentControlDataset(train_texts, tokenizer)
val_dataset = SentimentControlDataset(val_texts, tokenizer)

def fine_tune_sentiment_model(model, train_dataset, val_dataset, batch_size=8, epochs=3):
    """Fine-tunes GPT-2 on sentiment-controlled dataset."""

    # Move model to GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    training_args = TrainingArguments(
        output_dir="./gpt2_sentiment_control",
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        save_steps=500,
        save_total_limit=2,
        logging_dir="./logs",
        logging_steps=50,
        learning_rate=5e-5,
        evaluation_strategy="epoch",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
    )

    trainer.train()
fine_tune_sentiment_model(model, train_dataset, val_dataset, batch_size=16, epochs=3)


#5 Evaluation / Generation
def evaluate_sentiment_control(model, tokenizer, prompts=None):
    """
    Generates text for a list of sentiment prompts
    Prints the results
    """
    if prompts is None:
        prompts = [
            "<sentiment> positive <text>",
            "<sentiment> negative <text>"
        ]

    for prompt in prompts:
        print(f"\n=== Prompt: {prompt} ===")
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

        output = model.generate(
            input_ids,
            max_length=50,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7,
            num_return_sequences=10  #Generate multiple samples
        )

        for i, generated in enumerate(output):
            text_out = tokenizer.decode(generated, skip_special_tokens=True)
            print(f"[Sample {i+1}]: {text_out}\n")

evaluate_sentiment_control(model, tokenizer)